In [17]:
"""
Stress test — emulates a Jupyter notebook workflow.
 
Cell 1 — init client
Cell 2 — long compute → large volume → send (fire and forget) → long compute continues
Cell 3 — long compute → large mesh   → send (fire and forget) → long compute continues
Cell 4 — wait for all sends, report timing
"""
 
import time
import logging
import numpy as np
 
from Client import DataClient
 
logging.basicConfig(level=logging.INFO)
logging.getLogger("DataClient").setLevel(logging.DEBUG)

In [18]:
def fake_compute(label: str, seconds: float) -> np.ndarray:
    """Emulates expensive computation — burns CPU doing real work."""
    print(f"  [{label}] computing for {seconds:.1f}s ...")
    t0     = time.perf_counter()
    result = 0.0
    # burn real CPU so the background send truly runs in parallel
    while time.perf_counter() - t0 < seconds:
        result += np.sum(np.random.rand(512, 512))
    elapsed = time.perf_counter() - t0
    print(f"  [{label}] done  ({elapsed:.2f}s)")
    return result
 
 
def make_volume(shape: tuple[int, ...], dtype="float32") -> np.ndarray:
    print(f"  [make_volume] allocating {shape}  dtype={dtype} ...")
    t0  = time.perf_counter()
    arr = np.random.rand(*shape).astype(dtype)
    print(f"  [make_volume] done  ({time.perf_counter() - t0:.2f}s)  "
          f"size={arr.nbytes / 1024**2:.1f} MB")
    return arr
 
 
def make_mesh(n_vertices: int, n_faces: int):
    print(f"  [make_mesh] {n_vertices} vertices  {n_faces} faces ...")
    t0       = time.perf_counter()
    vertices = np.random.rand(n_vertices, 3).astype("float32")
    faces    = np.random.randint(0, n_vertices, (n_faces, 3), dtype="int32")
    print(f"  [make_mesh] done  ({time.perf_counter() - t0:.2f}s)  "
          f"v={vertices.nbytes/1024**2:.1f}MB  f={faces.nbytes/1024**2:.1f}MB")
    return vertices, faces
 
 
def align(size: int, chunk: int) -> int:
    """Round size up to the nearest multiple of chunk."""
    return ((size + chunk - 1) // chunk) * chunk

In [19]:
print("=" * 60)
print("CELL 1 — connect client")
print("=" * 60)
 
t_start = time.perf_counter()
client  = DataClient("localhost:50051")
print(f"Client ready  ({time.perf_counter() - t_start:.2f}s)\n")

INFO:Client.DataClient:DataClient connected to localhost:50051


CELL 1 — connect client
Client ready  (0.00s)



In [20]:
from typing import Any
print("=" * 60)
print("CELL 2 — long compute → send volume → continue computing")
print("=" * 60)
 
t_cell2 = time.perf_counter()
 
# Part A: long computation that produces a large volume
fake_compute("pre-volume", seconds=3.0)
 
CHUNK = 32
shape = (
    align(128, CHUNK),
    align(128, CHUNK),
    align(128, CHUNK),
)   # 128^3 float32 = 8 MB
vol: np.ndarray[tuple[Any, ...], np.dtype[Any]]  = make_volume(shape)
 
print(f"\n  [cell2] firing send_volume — will NOT block ...")
t_send = time.perf_counter()
client.send_volume("vol_stress_001", vol, chunks=(CHUNK, CHUNK, CHUNK))
print(f"  [cell2] send_volume returned in {time.perf_counter() - t_send:.4f}s  "
      f"← should be near zero\n")
 
# Part B: long computation AFTER the send — runs while transfer happens in background
fake_compute("post-volume", seconds=5.0)
 
print(f"  [cell2] total wall time: {time.perf_counter() - t_cell2:.2f}s\n")
 

CELL 2 — long compute → send volume → continue computing
  [pre-volume] computing for 3.0s ...
  [pre-volume] done  (3.00s)
  [make_volume] allocating (128, 128, 128)  dtype=float32 ...
  [make_volume] done  (0.01s)  size=8.0 MB

  [cell2] firing send_volume — will NOT block ...
  [cell2] send_volume returned in 0.0002s  ← should be near zero

  [post-volume] computing for 5.0s ...


INFO:Client.DataClient:Volume 'vol_stress_001' sent successfully


  [post-volume] done  (5.00s)
  [cell2] total wall time: 8.01s



In [21]:
print("=" * 60)
print("CELL 3 — long compute → send mesh → continue computing")
print("=" * 60)
 
t_cell3 = time.perf_counter()
 
fake_compute("pre-mesh", seconds=2.0)
 
V_CHUNK = 512
F_CHUNK = 512
n_verts = align(10_000, V_CHUNK)   # 10k vertices, float32 (N,3) = ~120KB
n_faces = align(5_000,  F_CHUNK)   # 5k  faces,    int32  (M,3) = ~60KB
 
vertices, faces = make_mesh(n_verts, n_faces)
 
print(f"\n  [cell3] firing send_mesh — will NOT block ...")
t_send = time.perf_counter()
client.send_mesh(
    "mesh_stress_001",
    vertices, faces,
    v_chunks=(V_CHUNK, 3),
    f_chunks=(F_CHUNK, 3),
)
print(f"  [cell3] send_mesh returned in {time.perf_counter() - t_send:.4f}s  "
      f"← should be near zero\n")
 
fake_compute("post-mesh", seconds=4.0)
 
print(f"  [cell3] total wall time: {time.perf_counter() - t_cell3:.2f}s\n")

CELL 3 — long compute → send mesh → continue computing
  [pre-mesh] computing for 2.0s ...


INFO:Client.DataClient:Mesh 'mesh_stress_001' sent successfully


  [pre-mesh] done  (2.00s)
  [make_mesh] 10240 vertices  5120 faces ...
  [make_mesh] done  (0.00s)  v=0.1MB  f=0.1MB

  [cell3] firing send_mesh — will NOT block ...
  [cell3] send_mesh returned in 0.0002s  ← should be near zero

  [post-mesh] computing for 4.0s ...
  [post-mesh] done  (4.00s)
  [cell3] total wall time: 6.00s



In [22]:
print("=" * 60)
print("CELL 4 — fire 3 volumes concurrently")
print("=" * 60)
 
t_cell4 = time.perf_counter()
 
for i in range(3):
    v = make_volume((align(64, CHUNK), align(64, CHUNK), align(64, CHUNK)))
    client.send_volume(f"vol_stress_00{i+2}", v, chunks=(CHUNK, CHUNK, CHUNK))
    print(f"  vol_stress_00{i+2} queued")
 
print(f"\n  All 3 queued in {time.perf_counter() - t_cell4:.2f}s  "
      f"← main thread never blocked\n")
 

CELL 4 — fire 3 volumes concurrently
  [make_volume] allocating (64, 64, 64)  dtype=float32 ...
  [make_volume] done  (0.00s)  size=1.0 MB
  vol_stress_002 queued
  [make_volume] allocating (64, 64, 64)  dtype=float32 ...
  [make_volume] done  (0.00s)  size=1.0 MB
  vol_stress_003 queued
  [make_volume] allocating (64, 64, 64)  dtype=float32 ...
  [make_volume] done  (0.01s)  size=1.0 MB
  vol_stress_004 queued

  All 3 queued in 0.01s  ← main thread never blocked



In [23]:
print("=" * 60)
print("CELL 5 — wait() for all pending sends")
print("=" * 60)
 
t_wait = time.perf_counter()
client.wait()
elapsed = time.perf_counter() - t_wait
 
if elapsed < 0.1:
    print(f"  All sends already done before wait() was called ({elapsed:.3f}s)")
else:
    print(f"  Waited {elapsed:.2f}s for remaining sends to complete")
 
print(f"\n  Total script time: {time.perf_counter() - t_start:.2f}s")

INFO:Client.DataClient:Volume 'vol_stress_002' sent successfully
INFO:Client.DataClient:Volume 'vol_stress_004' sent successfully
INFO:Client.DataClient:Volume 'vol_stress_003' sent successfully


CELL 5 — wait() for all pending sends
  All sends already done before wait() was called (0.083s)

  Total script time: 14.14s


In [24]:
print("\n" + "=" * 60)
print("CELL 6 — close client")
print("=" * 60)
 
client.close()
print("Done.")

INFO:Client.DataClient:DataClient closed



CELL 6 — close client
Done.
